## Stage 1: Environment Setup 

In [1]:

import pandas as pd
from pathlib import Path


project_root = Path.cwd().parent

processed_csv_path = project_root / "data" / "processed"

csv_files = list(processed_csv_path.glob("*.csv"))



if len(csv_files) == 1:
    csv_path = csv_files[0]
    print(f"\nSelected CSV: {csv_path}")
    print(f"File exists: {processed_csv_path.exists()}")

    df = pd.read_csv(csv_path)

    print(f"\nShape of processed dataframe: {df.shape}")

    print(f"\nColumn names: \n {df.columns.tolist()}")

else:
    raise FileNotFoundError(
        f"Expected exactly 1 CSV file in {processed_csv_path}, but found {len(csv_files)}. "
        "Check your data/processed folder before continuing."
    )

# !pip install duckdb                   #------------------> If user do not have package installed uncomment this line !

# !pip install jupysql duckdb-engine    #------------------> same as before if not installed !

import duckdb 

%load_ext sql

%sql duckdb:///:memory:

print("\nSQL Engine is active!")

%config SqlMagic.displaylimit = 100



Selected CSV: c:\Users\soumo\agy2-projects\ITSM-Agent-Capstone\data\processed\incidents_level_df.csv
File exists: True

Shape of processed dataframe: (24918, 29)

Column names: 
 ['number', 'incident_state', 'active', 'made_sla', 'opened_at', 'sys_updated_at', 'resolved_at', 'closed_at', 'priority', 'impact', 'urgency', 'category', 'subcategory', 'assignment_group', 'assigned_to', 'problem_id', 'change', 'caused_by', 'closed_code', 'reassignment_count', 'reopen_count', 'sys_mod_count', 'resolution_hours', 'is_sla_breached', 'has_problem_link', 'has_change_link', 'has_caused_by_link', 'is_reopened', 'was_reassigned']


Connecting to 'duckdb:///:memory:'


SQL Engine is active!


# Stage 2 : Data Loading & Validation


In [2]:
%%sql

CREATE OR REPLACE TABLE incidents AS SELECT * FROM '{{csv_path}}';

SELECT * FROM incidents LIMIT 6;



Running query in 'duckdb:///:memory:'

number,incident_state,active,made_sla,opened_at,sys_updated_at,resolved_at,closed_at,priority,impact,urgency,category,subcategory,assignment_group,assigned_to,problem_id,change,caused_by,closed_code,reassignment_count,reopen_count,sys_mod_count,resolution_hours,is_sla_breached,has_problem_link,has_change_link,has_caused_by_link,is_reopened,was_reassigned
INC0000045,Closed,False,True,2016-02-29 01:16:00,2016-03-05 12:00:00,2016-02-29 11:29:00,2016-03-05 12:00:00,3 - Moderate,2 - Medium,2 - Medium,Category 55,Subcategory 170,Group 56,None,None,None,None,code 5,0,0,4,10.216666666666667,False,False,False,False,False,False
INC0000047,Closed,False,True,2016-02-29 04:40:00,2016-03-06 10:00:00,2016-03-01 09:52:00,2016-03-06 10:00:00,3 - Moderate,2 - Medium,2 - Medium,Category 40,Subcategory 215,Group 24,Resolver 89,None,None,None,code 5,1,0,8,29.2,False,False,False,False,False,True
INC0000057,Closed,False,True,2016-02-29 06:10:00,2016-03-06 03:00:00,2016-03-01 02:55:00,2016-03-06 03:00:00,3 - Moderate,2 - Medium,2 - Medium,Category 20,Subcategory 125,Group 70,Resolver 6,Problem ID 2,None,None,code 10,0,0,6,20.75,False,True,False,False,False,False
INC0000060,Closed,False,True,2016-02-29 06:38:00,2016-03-07 13:00:00,2016-03-02 12:06:00,2016-03-07 13:00:00,3 - Moderate,2 - Medium,2 - Medium,Category 9,Subcategory 97,Group 25,Resolver 125,None,None,None,code 3,0,0,3,53.46666666666667,False,False,False,False,False,False
INC0000062,Closed,False,False,2016-02-29 06:58:00,2016-03-05 16:00:00,2016-02-29 15:51:00,2016-03-05 16:00:00,2 - High,1 - High,2 - Medium,Category 53,Subcategory 168,Group 23,None,None,None,None,code 7,1,0,7,8.883333333333333,True,False,False,False,False,True
INC0000063,Closed,False,True,2016-02-29 07:08:00,2016-03-05 17:00:00,2016-02-29 16:01:00,2016-03-05 17:00:00,3 - Moderate,2 - Medium,2 - Medium,Category 20,Subcategory 125,Group 23,None,None,None,None,code 7,1,0,7,8.883333333333333,False,False,False,False,False,True


In [3]:
%%sql
SELECT 
    COUNT(*) AS total_records, 
    COUNT(DISTINCT number) AS unique_incident_numbers,
    COUNT(*) = COUNT(DISTINCT number) AS one_row_per_incident
FROM incidents;

Running query in 'duckdb:///:memory:'

total_records,unique_incident_numbers,one_row_per_incident
24918,24918,True


In [4]:
%%sql
SUMMARIZE incidents;

Running query in 'duckdb:///:memory:'

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
number,VARCHAR,INC0000045,INC0121064,29237,None,None,None,None,None,24918,0.00
incident_state,VARCHAR,Closed,Closed,1,None,None,None,None,None,24918,0.00
active,BOOLEAN,false,false,1,None,None,None,None,None,24918,0.00
made_sla,BOOLEAN,false,true,2,None,None,None,None,None,24918,0.00
opened_at,TIMESTAMP,2016-02-29 01:16:00,2017-02-16 14:17:00,17835,2016-04-15 16:17:42.879846,None,2016-03-21 18:45:28.418426,2016-04-12 18:59:47.103457,2016-05-05 14:43:06.670308,24918,0.00
sys_updated_at,TIMESTAMP,2016-02-29 17:47:00,2017-02-18 15:00:00,2619,2016-04-28 19:37:25.961955,None,2016-04-02 19:39:37.573529,2016-04-23 05:25:51.580969,2016-05-16 14:27:25.626621,24918,0.00
resolved_at,TIMESTAMP,2016-02-29 09:04:00,2017-02-17 00:47:00,17570,2016-04-23 10:24:35.149388,None,2016-03-29 01:53:09.927255,2016-04-18 15:15:36.167783,2016-05-11 19:51:22.669333,24918,6.24
closed_at,TIMESTAMP,2016-02-29 17:47:00,2017-02-18 15:00:00,2619,2016-04-28 19:37:25.961955,None,2016-04-02 19:39:37.573529,2016-04-23 05:25:51.580969,2016-05-16 14:27:25.626621,24918,0.00
priority,VARCHAR,1 - Critical,4 - Low,4,None,None,None,None,None,24918,0.00
impact,VARCHAR,1 - High,3 - Low,3,None,None,None,None,None,24918,0.00


# Stage 3: SQL Business Query Foundations

In [5]:
%%sql

SELECT 
    column_name, 
    null_percentage
FROM (SUMMARIZE incidents)
WHERE null_percentage > 0.0
ORDER BY null_percentage DESC;

Running query in 'duckdb:///:memory:'

column_name,null_percentage
caused_by,99.99
change,99.29
problem_id,98.47
assignment_group,8.66
resolved_at,6.24
resolution_hours,6.24
assigned_to,2.91
closed_code,0.43
category,0.03
subcategory,0.03


In [6]:
%%sql
SELECT DISTINCT 
    priority AS unique_priority
FROM incidents
ORDER BY unique_priority;

Running query in 'duckdb:///:memory:'

unique_priority
1 - Critical
2 - High
3 - Moderate
4 - Low


In [7]:
%%sql
SELECT
    COUNT(*) AS critical_missing_assignment_group_count
FROM incidents
WHERE assignment_group IS NULL
  AND priority = '1 - Critical';
  

Running query in 'duckdb:///:memory:'

critical_missing_assignment_group_count
5


In [8]:
%%sql
SELECT
    number AS critical_missing_assignment_group_tickets
FROM incidents
WHERE assignment_group IS NULL 
    AND priority = '1 - Critical';

Running query in 'duckdb:///:memory:'

critical_missing_assignment_group_tickets
INC0003092
INC0006893
INC0011791
INC0021899
INC0025218


In [9]:
%%sql

SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE resolution_hours IS NOT NULL) AS valid_resolution_rows,
    COUNT(*) FILTER (WHERE resolution_hours IS NULL) AS missing_resolution_rows
FROM incidents;

Running query in 'duckdb:///:memory:'

total_rows,valid_resolution_rows,missing_resolution_rows
24918,23362,1556


In [10]:
%%sql

SELECT DISTINCT
    impact AS unique_impact
FROM incidents
ORDER BY unique_impact;

Running query in 'duckdb:///:memory:'

unique_impact
1 - High
2 - Medium
3 - Low


In [11]:
%%sql

SELECT DISTINCT
    urgency AS unique_urgency
FROM incidents
ORDER BY unique_urgency;

Running query in 'duckdb:///:memory:'

unique_urgency
1 - High
2 - Medium
3 - Low


In [12]:
%%sql

SELECT DISTINCT
    incident_state AS unique_incident_state
FROM incidents
ORDER BY unique_incident_state;

Running query in 'duckdb:///:memory:'

unique_incident_state
Closed


# Stage 4: Business KPI and Diagnostic SQL Analysis

## 4.1 Core Volume and Distribution KPIs

### 4.1.1 : How many total incidents exist in the dataset?


In [13]:
%%sql

SELECT 
    COUNT(*) AS total_incidents
FROM incidents;

Running query in 'duckdb:///:memory:'

total_incidents
24918


### 4.1.2 : What is the distribution of incident states?


In [14]:
%%sql

SELECT 
    incident_state,
    COUNT(*) AS incident_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS incident_percentage
FROM incidents
GROUP BY incident_state
ORDER BY incident_count DESC;

Running query in 'duckdb:///:memory:'

incident_state,incident_count,incident_percentage
Closed,24918,100.0


### 4.1.3: How are incidents distributed across impact levels?

In [15]:
%%sql

SELECT
    impact,
    COUNT(*) AS incident_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS incident_percentage
FROM incidents
GROUP BY impact
ORDER BY incident_count DESC;

Running query in 'duckdb:///:memory:'

impact,incident_count,incident_percentage
2 - Medium,23746,95.3
3 - Low,751,3.01
1 - High,421,1.69


### 4.1.4: How are incidents distributed across urgency levels?

In [16]:
%%sql

SELECT
    urgency,
    COUNT(*) AS incident_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS incident_percentage
FROM incidents
GROUP BY urgency
ORDER BY incident_count DESC;

Running query in 'duckdb:///:memory:'

urgency,incident_count,incident_percentage
2 - Medium,23689,95.07
3 - Low,698,2.8
1 - High,531,2.13


### 4.1.5: How are incidents distributed across priority levels?

In [17]:
%%sql

SELECT 
    priority,
    COUNT(*) AS incident_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS incident_percentage
FROM incidents
GROUP BY priority
ORDER BY incident_count DESC;

Running query in 'duckdb:///:memory:'

priority,incident_count,incident_percentage
3 - Moderate,23466,94.17
4 - Low,774,3.11
2 - High,408,1.64
1 - Critical,270,1.08


### 4.1.6 Top Categories by Incident Volume


In [18]:
%%sql

SELECT 
    category,
    COUNT(*) AS incident_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS incident_percentage
FROM incidents
WHERE category IS NOT NULL
GROUP BY category
ORDER BY incident_count DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

category,incident_count,incident_percentage
Category 42,3558,14.28
Category 26,3338,13.4
Category 53,2678,10.75
Category 46,2432,9.76
Category 32,1522,6.11
Category 9,1155,4.64
Category 37,1144,4.59
Category 23,1063,4.27
Category 20,1047,4.2
Category 57,971,3.9


### 4.1.7 Top Assignment Groups by Incident Volume


In [19]:
%%sql

SELECT 
    assignment_group,
    COUNT(*) AS incident_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS incident_percentage
FROM incidents
WHERE assignment_group IS NOT NULL
GROUP BY assignment_group
ORDER BY incident_count DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

assignment_group,incident_count,incident_percentage
Group 70,9444,41.49
Group 25,1243,5.46
Group 39,1199,5.27
Group 24,1060,4.66
Group 23,811,3.56
Group 64,716,3.15
Group 73,576,2.53
Group 28,545,2.39
Group 27,518,2.28
Group 20,394,1.73


## 4.2 SLA KPI Queries

### 4.2.1 Overall SLA Breach Distribution

In [71]:
%%sql

SELECT 
    is_sla_breached,
    COUNT(*) AS incident_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS incident_percentage
FROM incidents
GROUP BY is_sla_breached
ORDER BY incident_count DESC;

Running query in 'duckdb:///:memory:'

is_sla_breached,incident_count,incident_percentage
False,15803,63.42
True,9115,36.58


### 4.2.2 SLA Breached Incident Volume

In [72]:
%%sql

SELECT
    COUNT(*) AS sla_breached_incident_count
FROM incidents
WHERE is_sla_breached = true;

Running query in 'duckdb:///:memory:'

sla_breached_incident_count
9115


### 4.2.3 Overall SLA Breach Rate

In [73]:
%%sql

SELECT
    COUNT(*) AS total_incidents,
    COUNT(*) FILTER (WHERE is_sla_breached = true) AS breached_incidents,
    ROUND(
        COUNT(*) FILTER (WHERE is_sla_breached = true) * 100.0 / COUNT(*),
        2
    ) AS sla_breach_rate_percentage
FROM incidents;

Running query in 'duckdb:///:memory:'

total_incidents,breached_incidents,sla_breach_rate_percentage
24918,9115,36.58


### 4.2.4 SLA Breach Volume by Priority

In [74]:
%%sql

SELECT
    priority,
    COUNT(*) AS breached_incident_count
FROM incidents
WHERE is_sla_breached = true
GROUP BY priority
ORDER BY breached_incident_count DESC;

Running query in 'duckdb:///:memory:'

priority,breached_incident_count
3 - Moderate,8321
2 - High,406
1 - Critical,265
4 - Low,123


### 4.2.5 : SLA Breach rATE by Priority

In [55]:
%%sql

SELECT
    priority,
    COUNT(*) AS total_incidents,
    COUNT(*) FILTER (WHERE is_sla_breached = true) AS breached_incidents,
    ROUND(
        COUNT(*) FILTER (WHERE is_sla_breached = true) * 100.0 / COUNT(*),
        2
    ) AS breach_rate_percentage
FROM incidents
GROUP BY priority
ORDER BY breach_rate_percentage  DESC;

Running query in 'duckdb:///:memory:'

priority,total_incidents,breached_incidents,breach_rate_percentage
2 - High,408,406,99.51
1 - Critical,270,265,98.15
3 - Moderate,23466,8321,35.46
4 - Low,774,123,15.89


### 4.2.6 : Which impact level has the highest rate of SLA breaches?

In [23]:
%%sql

SELECT
    impact,
    COUNT(*) AS total_incidents,
    COUNT(*) FILTER (WHERE is_sla_breached = true) AS breached_incidents,
    ROUND(
        COUNT(*) FILTER (WHERE is_sla_breached = true) * 100.0 / COUNT(*),
        2
    ) AS breach_rate_percentage
   
FROM incidents
GROUP BY impact
ORDER BY breach_rate_percentage DESC;

Running query in 'duckdb:///:memory:'

impact,total_incidents,breached_incidents,breach_rate_percentage
1 - High,421,416,98.81
2 - Medium,23746,8591,36.18
3 - Low,751,108,14.38


### 4.2.7 : Which urgency level has the highest SLA breach rate?

In [24]:
%%sql

SELECT
    urgency,
    COUNT(*) AS total_incidents,
    COUNT(*) FILTER (WHERE is_sla_breached = true) AS breached_incidents,
    ROUND(
        COUNT(*) FILTER (WHERE is_sla_breached = true) * 100.0 / COUNT(*),
        2
    ) AS breach_rate_percentage
   
FROM incidents
GROUP BY urgency
ORDER BY breach_rate_percentage DESC;

Running query in 'duckdb:///:memory:'

urgency,total_incidents,breached_incidents,breach_rate_percentage
1 - High,531,521,98.12
2 - Medium,23689,8484,35.81
3 - Low,698,110,15.76


### 4.2.8 : Which categories have the highest SLA breach rate?

In [58]:
%%sql

SELECT
    category,
    COUNT(*) AS total_incidents,
    COUNT(*) FILTER (WHERE is_sla_breached = true) AS breached_incidents,
    ROUND(
        COUNT(*) FILTER (WHERE is_sla_breached = true) * 100.0 / COUNT(*),
        2
    ) AS breach_rate_percentage
FROM incidents
WHERE category IS NOT NULL
GROUP BY category
HAVING COUNT(*) >= 100
ORDER BY breach_rate_percentage DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

category,total_incidents,breached_incidents,breach_rate_percentage
Category 55,106,84,79.25
Category 45,602,446,74.09
Category 34,501,333,66.47
Category 40,436,266,61.01
Category 19,245,130,53.06
Category 61,810,429,52.96
Category 46,2432,1254,51.56
Category 44,302,151,50.0
Category 23,1063,505,47.51
Category 24,640,303,47.34


### 4.2.9 : Which assignment groups have the highest SLA breach rate?

In [59]:
%%sql

SELECT
    assignment_group,
    COUNT(*) AS total_incidents,
    COUNT(*) FILTER (WHERE is_sla_breached = true) AS breached_incidents,
    ROUND(
        COUNT(*) FILTER (WHERE is_sla_breached = true) * 100.0 / COUNT(*),
        2
    ) AS breach_rate_percentage
FROM incidents
WHERE assignment_group IS NOT NULL
GROUP BY assignment_group
HAVING COUNT(*) >= 100
ORDER BY breach_rate_percentage DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

assignment_group,total_incidents,breached_incidents,breach_rate_percentage
Group 12,123,107,86.99
Group 37,176,132,75.0
Group 10,343,257,74.93
Group 66,376,262,69.68
Group 74,105,73,69.52
Group 31,205,140,68.29
Group 57,313,213,68.05
Group 72,367,232,63.22
Group 29,257,160,62.26
Group 33,201,123,61.19


## 4.3 Reopen KPI and Diagnostic Queries


### 4.3.1 Reopened Incident Volume

In [27]:
%%sql

SELECT
    COUNT(*) AS reopened_incident_count
FROM incidents
WHERE is_reopened = true;

Running query in 'duckdb:///:memory:'

reopened_incident_count
275


### 4.3.2 Overall Reopen Distribution

In [28]:
%%sql

SELECT 
    is_reopened,
    COUNT(*) AS incident_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS incident_percentage
FROM incidents
GROUP BY is_reopened
ORDER BY incident_count DESC;

Running query in 'duckdb:///:memory:'

is_reopened,incident_count,incident_percentage
False,24643,98.9
True,275,1.1


### 4.3.3 : Among categories with meaningful incident volume, which categories have the highest reopen rate?

In [60]:
%%sql

SELECT
    category,
    COUNT(*) AS total_incidents,
    COUNT(*) FILTER (WHERE is_reopened = true) AS reopened_incidents,
    ROUND(
        COUNT(*) FILTER (WHERE is_reopened = true) * 100.0 / COUNT(*),
        2
    ) AS reopen_rate_percentage
FROM incidents
WHERE category IS NOT NULL
GROUP BY category
HAVING COUNT(*) >= 100
ORDER BY reopen_rate_percentage DESC, reopened_incidents DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

category,total_incidents,reopened_incidents,reopen_rate_percentage
Category 61,810,19,2.35
Category 57,971,17,1.75
Category 9,1155,17,1.47
Category 53,2678,38,1.42
Category 40,436,6,1.38
Category 43,153,2,1.31
Category 32,1522,19,1.25
Category 26,3338,39,1.17
Category 20,1047,12,1.15
Category 23,1063,12,1.13


### 4.3.4 : Among assignment groups with meaningful incident volume, which groups have the highest reopen rate?

In [61]:
%%sql

SELECT
    assignment_group,
    COUNT(*) AS total_incidents,
    COUNT(*) FILTER (WHERE is_reopened = true) AS reopened_incidents,
    ROUND(
        COUNT(*) FILTER (WHERE is_reopened = true) * 100.0 / COUNT(*) , 2
    ) AS reopen_rate_percentage
FROM incidents
WHERE assignment_group IS NOT NULL
GROUP BY assignment_group
HAVING COUNT(*) >= 100
ORDER BY reopen_rate_percentage DESC, reopened_incidents DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

assignment_group,total_incidents,reopened_incidents,reopen_rate_percentage
Group 74,105,3,2.86
Group 39,1199,32,2.67
Group 29,257,5,1.95
Group 73,576,11,1.91
Group 56,167,3,1.8
Group 27,518,9,1.74
Group 25,1243,21,1.69
Group 22,239,4,1.67
Group 28,545,9,1.65
Group 54,200,3,1.5


## 4.4 Reassignment KPI and Diagnostic Queries


### 4.4.1 : What is the overall incident reassignment distribution?

In [31]:
%%sql

SELECT 
    was_reassigned,
    COUNT(*) AS incident_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS incident_percentage
FROM incidents
GROUP BY was_reassigned
ORDER BY incident_count DESC;

Running query in 'duckdb:///:memory:'

was_reassigned,incident_count,incident_percentage
False,13549,54.37
True,11369,45.63


### 4.4.2 : What is the total volume of reassigned incidents?

In [32]:
%%sql

SELECT
    COUNT(*) AS reassigned_incident_count
FROM incidents
WHERE was_reassigned = true;

Running query in 'duckdb:///:memory:'

reassigned_incident_count
11369


### 4.4.3 Reassigned Incident Volume by Assignment Group

In [62]:
%%sql

SELECT
    assignment_group,
    COUNT(*) AS reassigned_incident_count
FROM incidents
WHERE was_reassigned = true
  AND assignment_group IS NOT NULL
GROUP BY assignment_group
ORDER BY reassigned_incident_count DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

assignment_group,reassigned_incident_count
Group 70,1991
Group 39,934
Group 25,719
Group 23,573
Group 24,572
Group 28,347
Group 73,343
Group 72,297
Group 20,281
Group 27,278


### 4.4.4 Reassignment Rate by Assignment Group

In [33]:
%%sql

SELECT
    assignment_group,
    COUNT(*) AS total_incidents,
    COUNT(*) FILTER (WHERE was_reassigned = true) AS reassigned_incidents,
    ROUND(
        COUNT(*) FILTER (WHERE was_reassigned = true) * 100.0 / COUNT(*),
        2
    ) AS reassignment_rate_percentage
FROM incidents
WHERE assignment_group IS NOT NULL
GROUP BY assignment_group
HAVING COUNT(*) >= 100
ORDER BY reassignment_rate_percentage DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

assignment_group,total_incidents,reassigned_incidents,reassignment_rate_percentage
Group 56,167,138,82.63
Group 72,367,297,80.93
Group 39,1199,934,77.9
Group 54,200,150,75.0
Group 74,105,78,74.29
Group 20,394,281,71.32
Group 23,811,573,70.65
Group 6,257,174,67.7
Group 28,545,347,63.67
Group 30,268,162,60.45


##  4.5 Resolution-Time Diagnostic Queries

These queries measure how long incidents take to resolve using both average and median resolution hours. The flow starts with the overall resolution-time baseline, then moves into priority, impact, urgency, category, and assignment-group diagnostics.

### 4.5.1 : What is the overall average and median resolution time?

In [34]:
%%sql

SELECT
    COUNT(*) AS total_resolved_incidents,
    ROUND(AVG(resolution_hours), 2) AS overall_avg_resolution_hours,
    ROUND(MEDIAN(resolution_hours), 2) AS overall_median_resolution_hours
FROM incidents
WHERE resolution_hours IS NOT NULL;

Running query in 'duckdb:///:memory:'

total_resolved_incidents,overall_avg_resolution_hours,overall_median_resolution_hours
23362,178.17,22.1


### 4.5.2 : How fast are incidents resolved by priority, and are extreme outliers skewing average resolution time?

In [35]:
%%sql

SELECT
    priority,
    COUNT(*) AS resolved_incidents,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    ROUND(MEDIAN(resolution_hours), 2) AS median_resolution_hours
FROM incidents
WHERE resolution_hours IS NOT NULL
GROUP BY priority
ORDER BY avg_resolution_hours DESC;

Running query in 'duckdb:///:memory:'

priority,resolved_incidents,avg_resolution_hours,median_resolution_hours
4 - Low,674,283.19,5.02
1 - Critical,270,266.08,80.24
3 - Moderate,22010,174.35,21.55
2 - High,408,152.46,38.23


### 4.5.3 : How fast are incidents resolved by impact, and are extreme outliers skewing average resolution time?

In [36]:
%%sql

SELECT
    impact,
    COUNT(*) AS resolved_incidents,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    ROUND(MEDIAN(resolution_hours), 2) AS median_resolution_hours
FROM incidents
WHERE resolution_hours IS NOT NULL
GROUP BY impact
ORDER BY avg_resolution_hours DESC;

Running query in 'duckdb:///:memory:'

impact,resolved_incidents,avg_resolution_hours,median_resolution_hours
3 - Low,651,262.97,4.18
1 - High,421,216.3,54.22
2 - Medium,22290,174.97,21.88


### 4.5.4 : How fast are incidents resolved by urgency, and are extreme outliers skewing average resolution time?

In [37]:
%%sql

SELECT
    urgency,
    COUNT(*) AS resolved_incidents,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    ROUND(MEDIAN(resolution_hours), 2) AS median_resolution_hours
FROM incidents
WHERE resolution_hours IS NOT NULL
GROUP BY urgency
ORDER BY avg_resolution_hours DESC;

Running query in 'duckdb:///:memory:'

urgency,resolved_incidents,avg_resolution_hours,median_resolution_hours
3 - Low,607,273.14,3.7
1 - High,531,218.54,69.12
2 - Medium,22224,174.61,21.72


### 4.5.5 : Which categories have the highest average and median resolution hours?

In [63]:
%%sql

SELECT
    category,
    COUNT(*) AS resolved_incident_count,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    ROUND(MEDIAN(resolution_hours), 2) AS median_resolution_hours
FROM incidents
WHERE resolution_hours IS NOT NULL
 AND category IS NOT NULL
GROUP BY category
HAVING COUNT(*) >= 100
ORDER BY avg_resolution_hours DESC , resolved_incident_count DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

category,resolved_incident_count,avg_resolution_hours,median_resolution_hours
Category 34,499,1326.32,404.72
Category 55,106,674.02,169.33
Category 45,589,481.75,147.73
Category 46,2356,316.02,65.71
Category 24,634,279.68,72.22
Category 19,244,277.65,61.12
Category 61,808,182.37,95.38
Category 57,961,161.84,42.58
Category 9,1121,156.23,45.77
Category 44,296,153.81,25.57


### 4.5.6 : Which assignment groups have the slowest average and median resolution times, and are outliers skewing their performance?

In [64]:
%%sql

SELECT
    assignment_group,
    COUNT(*) AS resolved_incident_count,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    ROUND(MEDIAN(resolution_hours), 2) AS median_resolution_hours
FROM incidents
WHERE resolution_hours IS NOT NULL
 AND assignment_group IS NOT NULL
GROUP BY assignment_group
HAVING COUNT(*) >= 100
ORDER BY avg_resolution_hours DESC, median_resolution_hours DESC 
LIMIT 10;

Running query in 'duckdb:///:memory:'

assignment_group,resolved_incident_count,avg_resolution_hours,median_resolution_hours
Group 31,205,758.3,101.12
Group 10,343,597.16,402.37
Group 66,374,498.59,207.77
Group 37,176,412.19,249.77
Group 33,201,406.82,114.5
Group 12,123,335.77,311.47
Group 57,313,312.57,167.15
Group 72,367,311.16,123.33
Group 6,257,305.25,120.85
Group 25,1243,251.94,111.7


## 4.6 Assignee and Resolver Workload Diagnostics

These queries analyze assignee-level workload, resolved-ticket volume, resolution-time patterns, and assignment-group resolver coverage. These are diagnostic queries, not direct employee performance judgments.

### 4.6.1 : Which assignees handled the most incidents overall?

In [66]:
%%sql

SELECT
    assigned_to,
    COUNT(*) AS incident_count
FROM incidents
WHERE assigned_to IS NOT NULL
AND resolution_hours IS NOT NULL
GROUP BY assigned_to
ORDER BY incident_count DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

assigned_to,incident_count
Resolver 13,2656
Resolver 17,2298
Resolver 194,676
Resolver 115,639
Resolver 73,484
Resolver 94,450
Resolver 57,423
Resolver 33,408
Resolver 69,405
Resolver 149,381


### 4.6.2 : Which assignees have the highest average and median resolution hours among assignees with meaningful resolved volume?

In [68]:
%%sql

SELECT
    assigned_to,
    COUNT(*) AS resolved_incident_count,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    ROUND(MEDIAN(resolution_hours), 2) AS median_resolution_hours
FROM incidents
WHERE assigned_to IS NOT NULL
AND resolution_hours IS NOT NULL
GROUP BY assigned_to
HAVING COUNT(*) >= 100
ORDER BY avg_resolution_hours DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

assigned_to,resolved_incident_count,avg_resolution_hours,median_resolution_hours
Resolver 217,101,1588.53,162.9
Resolver 200,167,613.52,404.02
Resolver 216,114,576.24,151.67
Resolver 34,184,516.56,51.09
Resolver 69,405,503.39,92.02
Resolver 153,174,460.36,315.48
Resolver 57,423,440.57,146.13
Resolver 182,144,420.81,94.84
Resolver 175,188,398.45,207.77
Resolver 101,145,350.5,165.83


### 4.6.3 : Which assignment groups have the largest observed resolver roster in the incident dataset?

In [42]:
%%sql

SELECT
    assignment_group,
    COUNT(DISTINCT assigned_to) AS resolver_count,
    LIST(DISTINCT assigned_to) AS roster_of_agents
FROM incidents
WHERE assignment_group IS NOT NULL
  AND assigned_to IS NOT NULL
GROUP BY assignment_group
ORDER BY resolver_count DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

assignment_group,resolver_count,roster_of_agents
Group 70,48,"['Resolver 115', 'Resolver 69', 'Resolver 27', 'Resolver 56', 'Resolver 233', 'Resolver 86', 'Resolver 175', 'Resolver 150', 'Resolver 191', 'Resolver 242', 'Resolver 6', 'Resolver 33', 'Resolver 13', 'Resolver 126', 'Resolver 15', 'Resolver 119', 'Resolver 205', 'Resolver 118', 'Resolver 213', 'Resolver 192', 'Resolver 42', 'Resolver 243', 'Resolver 216', 'Resolver 73', 'Resolver 215', 'Resolver 11', 'Resolver 19', 'Resolver 26', 'Resolver 186', 'Resolver 17', 'Resolver 28', 'Resolver 232', 'Resolver 57', 'Resolver 105', 'Resolver 31', 'Resolver 23', 'Resolver 47', 'Resolver 128', 'Resolver 253', 'Resolver 75', 'Resolver 127', 'Resolver 149', 'Resolver 208', 'Resolver 187', 'Resolver 30', 'Resolver 94', 'Resolver 218', 'Resolver 85']"
Group 20,20,"['Resolver 219', 'Resolver 224', 'Resolver 246', 'Resolver 69', 'Resolver 115', 'Resolver 233', 'Resolver 229', 'Resolver 93', 'Resolver 13', 'Resolver 33', 'Resolver 26', 'Resolver 232', 'Resolver 17', 'Resolver 188', 'Resolver 23', 'Resolver 90', 'Resolver 127', 'Resolver 30', 'Resolver 152', 'Resolver 94']"
Group 9,14,"['Resolver 66', 'Resolver 89', 'Resolver 111', 'Resolver 57', 'Resolver 140', 'Resolver 31', 'Resolver 34', 'Resolver 170', 'Resolver 217', 'Resolver 242', 'Resolver 119', 'Resolver 69', 'Resolver 44', 'Resolver 78']"
Group 5,12,"['Resolver 233', 'Resolver 33', 'Resolver 13', 'Resolver 15', 'Resolver 176', 'Resolver 94', 'Resolver 45', 'Resolver 28', 'Resolver 17', 'Resolver 150', 'Resolver 247', 'Resolver 237']"
Group 22,12,"['Resolver 13', 'Resolver 153', 'Resolver 133', 'Resolver 234', 'Resolver 208', 'Resolver 124', 'Resolver 94', 'Resolver 45', 'Resolver 155', 'Resolver 52', 'Resolver 26', 'Resolver 101']"
Group 65,12,"['Resolver 13', 'Resolver 41', 'Resolver 25', 'Resolver 7', 'Resolver 4', 'Resolver 180', 'Resolver 150', 'Resolver 2', 'Resolver 35', 'Resolver 210', 'Resolver 197', 'Resolver 23']"
Group 57,12,"['Resolver 155', 'Resolver 52', 'Resolver 153', 'Resolver 124', 'Resolver 40', 'Resolver 234', 'Resolver 133', 'Resolver 247', 'Resolver 45', 'Resolver 176', 'Resolver 101', 'Resolver 22']"
Group 55,12,"['Resolver 84', 'Resolver 56', 'Resolver 126', 'Resolver 33', 'Resolver 169', 'Resolver 214', 'Resolver 26', 'Resolver 134', 'Resolver 244', 'Resolver 228', 'Resolver 174', 'Resolver 30']"
Group 72,11,"['Resolver 112', 'Resolver 141', 'Resolver 250', 'Resolver 48', 'Resolver 123', 'Resolver 109', 'Resolver 110', 'Resolver 12', 'Resolver 116', 'Resolver 165', 'Resolver 98']"
Group 76,9,"['Resolver 23', 'Resolver 33', 'Resolver 241', 'Resolver 177', 'Resolver 108', 'Resolver 28', 'Resolver 17', 'Resolver 43', 'Resolver 208']"


## 4.7 Data-Quality Checks Supporting KPI Trust

These queries check missing or incomplete fields that affect the reliability of KPI, diagnostic, assignee, assignment-group, category, and resolution-time analysis.

### 4.7.1 : What is the volume of un-categorized incidents in our system?

In [43]:
%%sql

SELECT
    category,
    COUNT(*) AS missing_category_incidents
FROM incidents
WHERE category IS NULL
GROUP BY category;

Running query in 'duckdb:///:memory:'

category,missing_category_incidents
None,7


### 4.7.2 : How many incidents do not have an assignment group?

In [44]:
%%sql

SELECT
    assignment_group,
    COUNT(*) AS missing_assignment_group_incidents
FROM incidents
WHERE assignment_group IS NULL
GROUP BY assignment_group;

Running query in 'duckdb:///:memory:'

assignment_group,missing_assignment_group_incidents
None,2157


### 4.7.3 : What is the volume of reassigned incidents where the assignment group is NULL?

In [45]:
%%sql

SELECT
    assignment_group,
    COUNT(*) AS reassigned_incident_count
FROM incidents
WHERE was_reassigned = true
AND assignment_group IS NULL
GROUP BY assignment_group;

Running query in 'duckdb:///:memory:'

assignment_group,reassigned_incident_count
None,1791


### 4.7.4 : How many incidents have missing assignee values?

In [46]:
%%sql

SELECT
    COUNT(*) AS missing_assignee_incidents
FROM incidents
WHERE assigned_to IS NULL;

Running query in 'duckdb:///:memory:'

missing_assignee_incidents
725


### 4.7.5 : How many incidents are excluded from resolution-time metrics because resolution hours are missing?

In [47]:
%%sql
SELECT
    COUNT(*) AS incidents_missing_resolution_hours
FROM incidents
WHERE resolution_hours IS NULL;

Running query in 'duckdb:///:memory:'

incidents_missing_resolution_hours
1556


### 4.7.6 : How many incidents have missing closure code?

In [48]:
%%sql
SELECT
    COUNT(*) AS incidents_missing_closed_code
FROM incidents 
WHERE closed_code IS NULL;

Running query in 'duckdb:///:memory:'

incidents_missing_closed_code
107


##  4.8 Problem, Change, and Cause Linkage Diagnostics

These queries measure how many incidents are connected to problem records, change records, or caused-by-change records. This section helps the future agent explain whether incidents are isolated tickets or linked to broader problem/change-management processes.

### 4.8.1 : How many incidents are linked to a problem record?

In [69]:
%%sql

SELECT 
    COUNT(*) AS incidents_with_problems
FROM incidents
WHERE has_problem_link = true;


Running query in 'duckdb:///:memory:'

incidents_with_problems
380


### 4.8.2 : How many incidents are linked to a change record?

In [ ]:
%%sql

SELECT 
    COUNT(*) AS incidents_with_changes
FROM incidents
WHERE has_change_link = true;

Running query in 'duckdb:///:memory:'

incidents_with_changes
176


### 4.8.3 : How many incidents were caused by a change?

In [51]:
%%sql

SELECT 
    COUNT(*) AS incidents_caused_by_change,
    LIST(number) AS specific_ticket_numbers
FROM incidents
WHERE caused_by IS NOT NULL;

Running query in 'duckdb:///:memory:'

incidents_caused_by_change,specific_ticket_numbers
3,"['INC0000388', 'INC0001131', 'INC0031344']"


### 4.8.4 : What percentage of incidents have problem, change, or caused-by links?

In [70]:
%%sql

SELECT
    COUNT(*) AS total_incidents,

    COUNT(*) FILTER (WHERE has_problem_link = true) AS problem_linked_incidents,
    
    ROUND(
        COUNT(*) FILTER (WHERE has_problem_link = true) * 100.0 / COUNT(*), 
        2
    ) AS problem_link_percentage,
    
    COUNT(*) FILTER (WHERE has_change_link = true) AS change_linked_incidents,

    ROUND(
        COUNT(*) FILTER (WHERE has_change_link = true) * 100.0 / COUNT(*), 
        2
    ) AS change_link_percentage,
    
    COUNT(*) FILTER (WHERE has_caused_by_link = true) AS caused_by_change_incidents,

    ROUND(
        COUNT(*) FILTER (WHERE has_caused_by_link = true) * 100.0 / COUNT(*), 
        2
    ) AS caused_by_percentage

FROM incidents;

Running query in 'duckdb:///:memory:'

total_incidents,problem_linked_incidents,problem_link_percentage,change_linked_incidents,change_link_percentage,caused_by_change_incidents,caused_by_percentage
24918,380,1.53,176,0.71,3,0.01


## 4.9 SLA Breach Investigation Summary

This section summarizes the main operational patterns among SLA-breached incidents. It combines severity, category, assignment group, resolution time, reopen behavior, and reassignment behavior to support investigation-style agent responses.

This is not a root-cause proof. It is a pattern summary that helps identify where deeper investigation should begin.

### Before running the investigation query, we verify how many SLA-breached incidents remain after required fields are available.

In [53]:
%%sql

SELECT
    COUNT(*) AS total_incidents,
    COUNT(*) FILTER (WHERE is_sla_breached = true) AS sla_breached_incidents,
    COUNT(*) FILTER (
        WHERE is_sla_breached = true
          AND resolution_hours IS NOT NULL
    ) AS breached_with_resolution_hours,
    COUNT(*) FILTER (
        WHERE is_sla_breached = true
          AND resolution_hours IS NOT NULL
          AND category IS NOT NULL
          AND assignment_group IS NOT NULL
    ) AS breached_ready_for_investigation
FROM incidents;

Running query in 'duckdb:///:memory:'

total_incidents,sla_breached_incidents,breached_with_resolution_hours,breached_ready_for_investigation
24918,9115,8930,7821


### Main investigation SQL query

In [54]:
%%sql

WITH breached_base AS (
    SELECT
        priority,
        impact,
        urgency,
        category,
        assignment_group,
        resolution_hours,
        is_reopened,
        was_reassigned
    FROM incidents
    WHERE is_sla_breached = true
      AND resolution_hours IS NOT NULL
      AND category IS NOT NULL
      AND assignment_group IS NOT NULL
)

SELECT
    priority,
    impact,
    urgency,
    category,
    assignment_group,
    COUNT(*) AS breached_incidents,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM breached_base), 2) AS share_of_investigation_base_percentage,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    ROUND(MEDIAN(resolution_hours), 2) AS median_resolution_hours,
    COUNT(*) FILTER (WHERE is_reopened = true) AS reopened_incidents,
    ROUND(COUNT(*) FILTER (WHERE is_reopened = true) * 100.0 / COUNT(*), 2) AS reopen_rate_percentage,
    COUNT(*) FILTER (WHERE was_reassigned = true) AS reassigned_incidents,
    ROUND(COUNT(*) FILTER (WHERE was_reassigned = true) * 100.0 / COUNT(*), 2) AS reassignment_rate_percentage
FROM breached_base
GROUP BY
    priority,
    impact,
    urgency,
    category,
    assignment_group
HAVING COUNT(*) >= 50
ORDER BY
    breached_incidents DESC,
    avg_resolution_hours DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

priority,impact,urgency,category,assignment_group,breached_incidents,share_of_investigation_base_percentage,avg_resolution_hours,median_resolution_hours,reopened_incidents,reopen_rate_percentage,reassigned_incidents,reassignment_rate_percentage
3 - Moderate,2 - Medium,2 - Medium,Category 61,Group 73,219,2.8,235.77,198.25,6,2.74,174,79.45
3 - Moderate,2 - Medium,2 - Medium,Category 26,Group 70,201,2.57,274.05,199.25,13,6.47,141,70.15
3 - Moderate,2 - Medium,2 - Medium,Category 53,Group 70,184,2.35,280.0,244.22,12,6.52,159,86.41
3 - Moderate,2 - Medium,2 - Medium,Category 42,Group 39,148,1.89,187.38,127.84,11,7.43,138,93.24
3 - Moderate,2 - Medium,2 - Medium,Category 9,Group 25,143,1.83,272.3,235.6,4,2.8,49,34.27
3 - Moderate,2 - Medium,2 - Medium,Category 32,Group 70,136,1.74,223.32,167.43,3,2.21,68,50.0
3 - Moderate,2 - Medium,2 - Medium,Category 46,Group 10,127,1.62,797.6,693.2,1,0.79,72,56.69
3 - Moderate,2 - Medium,2 - Medium,Category 46,Group 57,126,1.61,399.27,324.68,0,0.0,64,50.79
3 - Moderate,2 - Medium,2 - Medium,Category 42,Group 70,119,1.52,278.87,167.62,3,2.52,62,52.1
3 - Moderate,2 - Medium,2 - Medium,Category 23,Group 70,111,1.42,330.82,293.3,6,5.41,100,90.09


# Stage 5 : Curated Business Query Catalog



## 5.1 Purpose of the Curated Agent Query Catalog

Stage 5 converts the SQL queries from Stage 4 into an agent-readable query catalog.

The purpose of this catalog is to help a future AI agent understand:

- which SQL query answers which business question
- what each query returns
- which metric family each query belongs to
- what business guardrails are already built into the SQL
- when the agent should use each query
- what caveats the agent must explain to the user

The SQL logic from Stage 4 is treated as the source of truth.  
This stage does not rewrite SQL. It documents the query layer so that the future agent can map user prompts to trusted query IDs.

## 5.2 Agent Query Contract

The future agent should follow this contract:

1. The agent should not write new SQL directly from the user prompt.
2. The agent should first map the user prompt to one approved `query_id`.
3. The agent should use only the SQL queries documented in this catalog.
4. The agent should explain results using business language, not raw SQL language.
5. The agent should respect each query's grain, filters, thresholds, and caveats.
6. The agent should not treat diagnostic queries as proof of root cause.
7. The agent should not treat assignee-level diagnostics as employee performance judgment.
8. If the user asks for a metric not available in the catalog, the agent should say the metric is not currently supported by this query layer.

## 5.3 Curated Query Catalog Index

This catalog lists the approved Stage 4 query IDs that the future agent can route user questions to.

The purpose of this cell is not to repeat every SQL output column.  
The purpose is to give the agent a clean map of:

- metric family
- query ID
- business question
- query type

The detailed routing rules are documented in the next cells.

| Metric Family | Query ID | Query Type | Business Question |
|---|---|---|---|
| Core Volume | `total_incidents` | KPI | How many total incidents exist? |
| Core Distribution | `incident_state_distribution` | Distribution | What is the distribution of incident states? |
| Core Distribution | `dist_by_impact` | Distribution | How are incidents distributed across impact levels? |
| Core Distribution | `dist_by_urgency` | Distribution | How are incidents distributed across urgency levels? |
| Core Distribution | `dist_by_priority` | Distribution | How are incidents distributed across priority levels? |
| Core Volume | `top_categories_by_volume` | Top-N Diagnostic | Which categories have the highest incident volume? |
| Core Volume | `top_assignment_groups_by_volume` | Top-N Diagnostic | Which assignment groups handled the most incidents? |
| SLA | `dist_overall_sla_breach` | Distribution | What is the overall SLA breach distribution? |
| SLA | `vol_sla_breached` | Volume KPI | How many incidents breached SLA? |
| SLA | `kpi_overall_sla_breach_rate` | KPI | What percentage of incidents breached SLA? |
| SLA | `vol_breach_by_priority` | Volume Diagnostic | How many breached incidents are in each priority level? |
| SLA | `kpi_breach_rate_by_priority` | Diagnostic KPI | Which priority level has the highest SLA breach rate? |
| SLA | `kpi_breach_rate_by_impact` | Diagnostic KPI | Which impact level has the highest SLA breach rate? |
| SLA | `kpi_breach_rate_by_urgency` | Diagnostic KPI | Which urgency level has the highest SLA breach rate? |
| SLA | `kpi_breach_rate_by_category` | Diagnostic KPI | Which categories have the highest SLA breach rate? |
| SLA | `kpi_breach_rate_by_assignment_group` | Diagnostic KPI | Which assignment groups have the highest SLA breach rate? |
| Reopen | `vol_reopened` | Volume KPI | How many incidents were reopened? |
| Reopen | `dist_overall_reopen` | Distribution | What is the overall reopen distribution? |
| Reopen | `kpi_reopen_rate_by_category` | Diagnostic KPI | Which categories have the highest reopen rate? |
| Reopen | `kpi_reopen_rate_by_assignment_group` | Diagnostic KPI | Which assignment groups have the highest reopen rate? |
| Reassignment | `dist_overall_reassignment` | Distribution | What is the overall reassignment distribution? |
| Reassignment | `vol_reassigned` | Volume KPI | How many incidents were reassigned? |
| Reassignment | `vol_reassigned_by_assignment_group` | Volume Diagnostic | Which assignment groups have the highest reassigned incident volume? |
| Reassignment | `kpi_reassign_rate_by_group` | Diagnostic KPI | Which assignment groups have the highest reassignment rate? |
| Resolution Time | `kpi_overall_resolution_time` | KPI | What is the overall average and median resolution time? |
| Resolution Time | `kpi_resolution_time_by_priority` | Diagnostic KPI | How fast are incidents resolved by priority? |
| Resolution Time | `kpi_resolution_time_by_impact` | Diagnostic KPI | How fast are incidents resolved by impact? |
| Resolution Time | `kpi_resolution_time_by_urgency` | Diagnostic KPI | How fast are incidents resolved by urgency? |
| Resolution Time | `kpi_resolution_time_by_category` | Diagnostic KPI | Which categories have the highest average and median resolution hours? |
| Resolution Time | `kpi_resolution_time_by_group` | Diagnostic KPI | Which assignment groups have the highest average and median resolution hours? |
| Assignee Workload | `assignee_workload_by_resolution_valid_incidents` | Workload Diagnostic | Which assignees handled the most resolution-valid incidents? |
| Assignee Resolution | `assignee_resolution_time_diagnostic` | Diagnostic | Which assignees have the highest average and median resolution hours? |
| Resolver Coverage | `assignment_group_observed_resolver_roster` | Operational Mapping | Which assignment groups have the largest observed resolver roster? |
| Data Quality | `data_missing_category` | Completeness Check | How many incidents have missing category values? |
| Data Quality | `data_missing_assignment_group` | Completeness Check | How many incidents have missing assignment group values? |
| Data Quality | `data_reassigned_missing_group` | Completeness Check | How many reassigned incidents have missing assignment group? |
| Data Quality | `data_missing_assignee` | Completeness Check | How many incidents have missing assignee values? |
| Data Quality | `data_missing_resolution_hours` | Completeness Check | How many incidents have missing resolution hours? |
| Data Quality | `data_missing_closed_code` | Completeness Check | How many incidents have missing closure code? |
| Linkage | `problem_linked_incidents` | Problem Diagnostic | How many incidents are linked to a problem record? |
| Linkage | `change_linked_incidents` | Change Diagnostic | How many incidents are linked to a change record? |
| Linkage | `caused_by_change_ticket_detail` | Cause Diagnostic | How many incidents were caused by a change? |
| Linkage | `linkage_summary` | Summary Diagnostic | What percentage of incidents have problem, change, or caused-by links? |
| SLA Investigation | `sla_breach_investigation_readiness_check` | Data Readiness Check | How many breached incidents are ready for investigation? |
| SLA Investigation | `investigation_summary_for_breached_sla` | Investigation Diagnostic | What patterns appear among SLA-breached incidents? |

### 5.3 Catalog Note

This table is the high-level query index.

It intentionally does not include SQL code, output columns, or long guardrail explanations.  
Those details remain in the Stage 4 SQL cells and the following Stage 5 routing/caveat sections.

For the future agent, this table answers one question:

**“Which approved query ID should I consider for this business question?”**

## 5.4 Natural Language Prompt Routing

The future agent should use this routing table to map user prompts to query IDs.

| User Prompt Pattern | Route to Query ID |
|---|---|
| "How many incidents are there?" | `total_incidents` |
| "Show incident state distribution" | `incident_state_distribution` |
| "Distribution by impact" | `dist_by_impact` |
| "Distribution by urgency" | `dist_by_urgency` |
| "Distribution by priority" | `dist_by_priority` |
| "Top incident categories" | `top_categories_by_volume` |
| "Top assignment groups by volume" | `top_assignment_groups_by_volume` |
| "How many incidents breached SLA?" | `vol_sla_breached` |
| "What percentage breached SLA?" | `kpi_overall_sla_breach_rate` |
| "SLA breach distribution" | `dist_overall_sla_breach` |
| "SLA breach by priority volume" | `vol_breach_by_priority` |
| "SLA breach rate by priority" | `kpi_breach_rate_by_priority` |
| "SLA breach rate by impact" | `kpi_breach_rate_by_impact` |
| "SLA breach rate by urgency" | `kpi_breach_rate_by_urgency` |
| "Which categories have high SLA breach rate?" | `kpi_breach_rate_by_category` |
| "Which groups have high SLA breach rate?" | `kpi_breach_rate_by_assignment_group` |
| "How many incidents were reopened?" | `vol_reopened` |
| "Reopen distribution" | `dist_overall_reopen` |
| "Reopen rate by category" | `kpi_reopen_rate_by_category` |
| "Reopen rate by assignment group" | `kpi_reopen_rate_by_assignment_group` |
| "How many incidents were reassigned?" | `vol_reassigned` |
| "Reassignment distribution" | `dist_overall_reassignment` |
| "Reassigned volume by assignment group" | `vol_reassigned_by_assignment_group` |
| "Reassignment rate by assignment group" | `kpi_reassign_rate_by_group` |
| "Overall resolution time" | `kpi_overall_resolution_time` |
| "Resolution time by priority" | `kpi_resolution_time_by_priority` |
| "Resolution time by impact" | `kpi_resolution_time_by_impact` |
| "Resolution time by urgency" | `kpi_resolution_time_by_urgency` |
| "Resolution time by category" | `kpi_resolution_time_by_category` |
| "Resolution time by assignment group" | `kpi_resolution_time_by_group` |
| "Which assignees handled the most tickets?" | `assignee_workload_by_resolution_valid_incidents` |
| "Which assignees have high resolution time?" | `assignee_resolution_time_diagnostic` |
| "Which groups have the largest resolver roster?" | `assignment_group_observed_resolver_roster` |
| "Missing category values" | `data_missing_category` |
| "Missing assignment group values" | `data_missing_assignment_group` |
| "Reassigned tickets with missing group" | `data_reassigned_missing_group` |
| "Missing assignee values" | `data_missing_assignee` |
| "Missing resolution hours" | `data_missing_resolution_hours` |
| "Missing closure code" | `data_missing_closed_code` |
| "Problem-linked incidents" | `problem_linked_incidents` |
| "Change-linked incidents" | `change_linked_incidents` |
| "Incidents caused by change" | `caused_by_change_ticket_detail` |
| "Problem/change linkage summary" | `linkage_summary` |
| "Can we investigate SLA breaches?" | `sla_breach_investigation_readiness_check` |
| "SLA breach investigation summary" | `investigation_summary_for_breached_sla` |

## 5.5 Ambiguity Handling Rules

The future agent should use these rules when a user prompt is unclear.

| Ambiguous User Request | Agent Decision Rule |
|---|---|
| "Show SLA performance" | Use `kpi_overall_sla_breach_rate` first. Then offer breach-rate breakdowns by priority, impact, urgency, category, or assignment group. |
| "Why are SLA breaches happening?" | Use `investigation_summary_for_breached_sla`. Explain that the output shows patterns, not proven root causes. |
| "Show bad categories" | Do not use negative wording. Map to `kpi_reopen_rate_by_category`, `kpi_breach_rate_by_category`, or `kpi_resolution_time_by_category` depending on the user's metric. |
| "Which team is worst?" | Do not answer as blame. Ask whether they mean SLA breach rate, reassignment rate, reopen rate, or resolution time by assignment group. |
| "Which assignee is best/worst?" | Do not judge performance. Use assignee diagnostics only as workload or resolution-pattern analysis. |
| "Show problem incidents" | Use `problem_linked_incidents` if the user wants count. Use `linkage_summary` if they want overall linkage view. |
| "Show change incidents" | Use `change_linked_incidents` if the user means linked to change. Use `caused_by_change_ticket_detail` if the user means caused by change. |
| "Show resolution performance" | Use `kpi_overall_resolution_time` first. Then route to priority, impact, urgency, category, or assignment group if a dimension is mentioned. |
| "Show data quality issues" | Use the `data_missing_*` queries from 4.7. |

## 5.6 Metric Family Logic

The future agent should understand the business purpose of each metric family.

| Metric Family | Purpose |
|---|---|
| Core Volume and Distribution | Establish the basic shape of the incident dataset. |
| SLA | Measure whether incidents met or breached service-level expectations. |
| Reopen | Identify incidents that required additional work after being resolved or closed. |
| Reassignment | Measure how often incidents moved across people or groups. |
| Resolution Time | Measure how long incidents took to resolve using both average and median. |
| Assignee and Resolver Workload | Analyze workload and resolution patterns at resolver level without judging employee performance. |
| Data Quality | Check missing fields that can weaken KPI trust. |
| Problem / Change / Cause Linkage | Identify whether incidents are connected to broader problem or change-management processes. |
| SLA Investigation | Summarize patterns inside breached-SLA incidents for deeper analysis. |

## 5.7 Caveats the Agent Must Explain

The future agent must explain these caveats when relevant.

| Query Area | Required Caveat |
|---|---|
| Resolution-time queries | Average can be affected by long-running outliers, so median should also be considered. |
| Category-level diagnostics | Category-level rankings exclude missing categories and often use top-N limits or minimum-volume thresholds. |
| Assignment-group diagnostics | Assignment-group results exclude missing groups where the SQL filter does so. |
| Assignee diagnostics | Assignee-level outputs are diagnostic only and should not be treated as employee performance proof. |
| Reopen and reassignment rates | Rate-based outputs are better for comparison than raw volume, but low-volume groups may still require caution. |
| Problem/change linkage | Linked counts depend on linkage fields being populated in the incident dataset. |
| Caused-by-change detail | The query returns a count and ticket-number list; the agent should summarize the count first and avoid overwhelming the user with ticket IDs unless requested. |
| SLA breach investigation | The investigation query shows patterns among breached incidents, not confirmed root causes. |
| Data-quality queries | Missing-field counts should be treated as limitations on downstream KPI interpretation. |

## 5.8 Recommended Agent Response Format

When the future agent answers a user question, it should structure the response like this:

1. **Selected Query ID**  
   State which approved query was used.

2. **Business Answer**  
   Explain the result in simple business language.

3. **Key Numbers**  
   Highlight the most important counts, percentages, rates, or time values.

4. **Caveat**  
   Mention any relevant limitation, such as missing values, outliers, minimum-volume threshold, or diagnostic-only interpretation.

5. **Suggested Follow-up**  
   Suggest one related query only if useful.

Example response pattern:

```text
Selected query: kpi_breach_rate_by_priority

Business answer:
High-priority incidents have the highest SLA breach rate.

Key numbers:
The query compares total incidents, breached incidents, and breach rate percentage for each priority level.

Caveat:
This shows breach-rate patterns by priority, not the root cause of the breaches.

Suggested follow-up:
Use investigation_summary_for_breached_sla to inspect common patterns among breached incidents.

## 5.9 Unsupported and Unsafe Prompt Handling

The future agent should not answer every user request by generating new SQL.

This query layer only supports questions that can be mapped to an approved `query_id` from the Stage 5 catalog. If a user asks for something outside the catalog, the agent should either:

1. route the question to the closest supported query,
2. ask a clarifying question, or
3. refuse the request if it is unsafe or impossible with the current dataset.

The agent must not invent unsupported metrics, query unknown tables, expose unrestricted row-level data, or perform destructive database actions.

---

### 5.9.1 Unsupported and Unsafe Prompt Categories

| Unsupported Prompt Type | Example User Prompt | Why It Is Unsupported | Agent Action |
|---|---|---|---|
| Unsupported metric | "Show CSAT by assignment group." | The current catalog has no customer satisfaction field or CSAT query. | Say the metric is not supported and offer available alternatives like SLA, reopen, reassignment, or resolution-time diagnostics. |
| Missing timestamp metric | "Calculate MTTA." | The current query catalog does not include acknowledgement or first-response timestamp logic. | Say MTTA is not available in this query layer. |
| Live backlog request | "Show current open backlog." | The current notebook is based on the processed incident dataset and Stage 4 query outputs, not a live operational backlog table. | Say backlog analysis is not supported in this current query layer. |
| Predictive request | "Predict which tickets will breach SLA next week." | The current notebook is descriptive and diagnostic, not predictive. | Say prediction is outside this SQL query layer and would require a separate modeling stage. |
| Unknown table request | "Query the users table." | The approved query layer only uses the `incidents` table. | Refuse and explain that only approved incident queries are supported. |
| Unknown column request | "Show customer region by SLA breach." | `customer_region` is not part of the current approved query catalog. | Refuse or ask for a supported dimension such as priority, impact, urgency, category, or assignment group. |
| Free-form SQL request | "Run this SQL: SELECT * FROM incidents." | The MVP agent should use approved query IDs, not arbitrary user-written SQL. | Refuse free-form execution and ask what business question the user wants answered. |
| Destructive SQL request | "Delete all records where SLA breached." | Delete, update, drop, truncate, or insert actions are unsafe and outside read-only analytics. | Refuse immediately. |
| Schema modification request | "Add a new column to the incidents table." | The query layer is read-only and should not modify schema. | Refuse and explain that the agent only supports analytical queries. |
| Unbounded row-level request | "Show every incident number in the dataset." | Large unrestricted row-level output is unsafe and not useful for KPI answering. | Refuse full dump; offer a summarized query or limited diagnostic output. |
| Sensitive/system request | "Show API keys or hidden credentials." | The incident query layer has no reason to access secrets or credentials. | Refuse immediately. |
| Blame-based employee request | "Who is the worst assignee?" | Assignee queries are diagnostic and cannot prove individual performance quality. | Reframe to workload or resolution-pattern diagnostics. |

---

### 5.9.2 Standard Agent Response Templates

Use these response patterns when a user request is unsupported, unsafe, unknown, ambiguous, or blame-risky.

#### Case 1: Unsupported metric

```text
I cannot answer that from the current approved query catalog because this metric is not available in the incident query layer.

Supported alternatives include SLA breach rate, reopen rate, reassignment rate, resolution time, assignment group diagnostics, assignee workload diagnostics, data-quality checks, and problem/change linkage summaries.
```

#### Case 2: Unsafe database action

```text
I cannot perform destructive or modifying database actions. This query layer is read-only and only supports approved analytical query IDs.
```

#### Case 3: Unknown table or column

```text
I cannot query that table or field because it is not part of the approved incidents query layer.

Please ask using supported dimensions such as priority, impact, urgency, category, assignment group, assignee, SLA breach, reopen status, reassignment status, resolution hours, or linkage flags.
```

#### Case 4: Ambiguous request

```text
I need to map your request to an approved query ID first.

Do you want SLA, reopen, reassignment, resolution-time, assignee, assignment-group, data-quality, or problem/change linkage analysis?
```

#### Case 5: Blame-based assignee request

```text
I cannot label an assignee as best or worst from this dataset.

I can provide workload or resolution-time diagnostics, but those results may reflect ticket complexity, routing, volume, or outliers rather than individual performance.
```

#### Case 6: Unsupported predictive request

```text
Prediction is outside the current SQL query catalog.

This notebook currently supports descriptive and diagnostic incident analytics. Predictive SLA breach modeling would need a separate modeling stage.
```

#### Case 7: Unbounded row-level request

```text
I cannot return unrestricted row-level incident dumps from this query layer.

I can provide a summarized KPI, top-N diagnostic, or a limited investigation output using an approved query ID.
```

---

### 5.9.3 Supported Alternatives the Agent Can Offer

When refusing an unsupported request, the agent should offer a nearby supported query whenever possible.

| If user asks for unsupported... | Agent can suggest... |
|---|---|
| CSAT or customer satisfaction | SLA breach rate, reopen rate, or resolution-time diagnostics |
| MTTA or first response time | Overall resolution time or resolution time by priority |
| Live backlog | Incident state distribution, if available in the processed dataset |
| Predictive SLA breach | SLA breach rate by priority, impact, urgency, category, or assignment group |
| Employee performance ranking | Assignee workload or assignee resolution-time diagnostic |
| Root cause proof | SLA breach investigation summary as pattern analysis, not root-cause proof |
| All raw incident rows | Total incident count, top categories, top assignment groups, or limited investigation summary |
| Unknown dimension | Supported dimensions: priority, impact, urgency, category, assignment group, assigned_to, SLA breach, reopen status, reassignment status, resolution hours, problem/change linkage |

---

### 5.9.4 Final Safety Rule

If a prompt cannot be mapped to an approved `query_id`, the agent should not generate SQL.

The correct behavior is:

```text
Unsupported request → explain why → offer the closest supported query
```

The agent should always prefer a safe refusal or clarification over invented SQL.